# MIDI parsing and df creation

## Import libraries

In [1]:
import os
from mido import MidiFile
import mido
import pandas as pd
import numpy as np

## Load MIDI file

In [2]:
def load_midi():
    
    # --- 1. mids/ ディレクトリを定義・作成 ---
    mids_dir = "mids"
    os.makedirs(mids_dir, exist_ok=True)
    # --- 修正完了 ---
    
    global midifile_path

    # --- 2. ファイル名のみを尋ねるように変更 ---
    mid_basename = input(f"Please insert the MIDI file name (must be in '{mids_dir}' folder) >> ")
    
    # --- 3. フルパスを自動で構築 ---
    midifile_path = os.path.join(mids_dir, mid_basename.strip()) # .strip() で前後の空白を削除

    def path_check(path):
        if os.path.exists(path):
            accept_ext = [".mid", ".midi", ".kar"]
            split_path = os.path.splitext(path)
            if split_path[1].lower() not in accept_ext: # .lower() で大文字の拡張子も許容
                print("Invalid file format. Please insert the path to a standard MIDI file: ")
                return False
            else:
                return True
        else:
            # --- 4. エラーメッセージを分かりやすく変更 ---
            print(f"Invalid path. File not found at: {path}")
            print(f"Please make sure '{mid_basename.strip()}' is inside the '{mids_dir}' folder.")
            return False

    # --- 5. ループ内の処理もファイル名入力 -> パス構築に変更 ---
    while not path_check(midifile_path):
        mid_basename = input(f"Please insert the MIDI file name (in '{mids_dir}' folder) >> ")
        midifile_path = os.path.join(mids_dir, mid_basename.strip())
    # --- 修正完了 ---

    print("Valid path.")
    print(f"Loading file: {midifile_path}")
    
load_midi()

Please insert the MIDI file name (must be in 'mids' folder) >>  complex_solo.mid


Valid path.
Loading file: mids/complex_solo.mid


## MIDI file df creation

In [3]:
import os
from mido import MidiFile
import mido
import pandas as pd
import numpy as np

# Note names - "s" for sharp - #
notes_names = ["C-1", "Cs-1", "D-1", "Ds-1", "E-1", "F-1", "Fs-1", "G-1", "Gs-1", "A-1", "As-1", "B-1", "C0", "Cs0", "D0", "Ds0", "E0", "F0", "Fs0", "G0", "Gs0", "A0", "As0", "B0", "C1", "Cs1", "D1", "Ds1", "E1", "F1", "Fs1", "G1", "Gs1", "A1", "As1", "B1", "C2", "Cs2", "D2", "Ds2", "E2", "F2", "Fs2", "G2", "Gs2", "A2", "As2", "B2", "C3", "Cs3", "D3", "Ds3", "E3", "F3", "Fs3", "G3", "Gs3", "A3", "As3", "B3", "C4", "Cs4", "D4", "Ds4", "E4", "F4", "Fs4", "G4", "Gs4", "A4", "As4", "B4", "C5", "Cs5", "D5", "Ds5", "E5", "F5", "Fs5", "G5", "Gs5", "A5", "As5", "B5", "C6", "Cs6", "D6", "Ds6", "E6", "F6", "Fs6", "G6", "Gs6", "A6", "As6", "B6", "C7", "Cs7", "D7", "Ds7", "E7", "F7", "Fs7", "G7", "Gs7", "A7", "As7", "B7", "C8", "Cs8", "D8", "Ds8", "E8", "F8", "Fs8", "G8", "Gs8", "A8", "As8", "B8", "C9", "Cs9", "D9", "Ds9", "E9", "F9", "Fs9", "G9"]
# --- ディレクトリ定義 ---
datas_dir = "datas" # .dat ファイルの「保存先」
results_dir = "results" 
os.makedirs(datas_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
# ---

def midifile_df_create():
    
    global notes_list_pd_df
    
    # 以前のセルで 'midifile_path' がグローバル変数として設定されている前提
    if 'midifile_path' not in globals():
        print("エラー: 'midifile_path' が設定されていません。")
        print("前の 'load_midi()' セルを先に実行してください。")
        return

    file_name_base = str(os.path.splitext(os.path.basename(midifile_path))[0])
    # (★ .dat ファイル名の重複回避ロジック ★)
    dat_output_base = os.path.join(datas_dir, file_name_base)
    dat_output_path = f"{dat_output_base}.dat"
    counter = 1
    while os.path.exists(dat_output_path):
        dat_output_path = f"{dat_output_base} ({counter}).dat"
        counter += 1
    
    dat_content_lines = [] 
    
    ##########################################################################
    #######   MIDI object creation and basic metamessages retrieval   ########
    ##########################################################################

    midifile = MidiFile(midifile_path)
    
    line = "// File name: " + file_name_base
    print(line); dat_content_lines.append(line)
    
    num_midi_tracks = len(midifile.tracks)
    line = "// Number of track in the MIDI file: " + str(num_midi_tracks)
    print(line); dat_content_lines.append(line)
    
    ticks_per_beat = midifile.ticks_per_beat
    line = "// Ticks per beat of the MIDI file: " + str(ticks_per_beat)
    print(line); dat_content_lines.append(line)
    
    tempo_found = False
    bpm = 120 
    tempo = 500000 
    for track in midifile.tracks:
        for msg in track:
            if msg.is_meta and msg.type == 'set_tempo':
                tempo = msg.tempo
                line = "// Tempo of the MIDI file: " + str(tempo)
                print(line); dat_content_lines.append(line)
                bpm = int(round(mido.tempo2bpm(tempo),2))
                tempo_found = True
                break
        if tempo_found: break
            
    time_sig_found = False
    numerator = 4 
    denominator = 4 
    for track in midifile.tracks:
        for msg in track:
             if msg.is_meta and msg.type == 'time_signature':
                numerator = msg.numerator
                denominator =  msg.denominator
                line = "// Time signature: " + str(numerator) + "/" + str(denominator) + "\\n"
                print(line); dat_content_lines.append(line)
                time_sig_found = True
                break
        if time_sig_found: break
            
    if not tempo_found:
        line = f"警告: MIDIにテンポ情報がありません。BPM={bpm} を仮定します。"
        print(line); dat_content_lines.append(f"// {line}")
    if not time_sig_found:
        line = f"警告: MIDIに拍子情報がありません。{numerator}/{denominator} を仮定します。\\n"
        print(line); dat_content_lines.append(f"// {line}")
    
    print("--- BPM確認 ---")
    bpm_input = input(f"検出されたBPM: {bpm} (このままEnter, または新しいBPMを入力) >> ")
    if bpm_input.strip(): 
        try:
            new_bpm = int(bpm_input.strip())
            print(f"BPMを {new_bpm} に上書きします。")
            bpm = new_bpm
            tempo = mido.bpm2tempo(bpm) 
            
            line = f"// Tempo (User Overridden): {tempo}"
            print(line); dat_content_lines.append(line)
                
        except ValueError:
            print(f"無効な入力です。元のBPM {bpm} を使用します。")
    
    line = f"// BPM: {bpm}"
    print(line); dat_content_lines.append(line)

    ##########################################################################
    ####################   Dataframe creation   ##############################
    ##########################################################################
    
    columns_names = ["note_numb", "note_name", "dt_note_on", "t_dt_note_on", "dt_note_off", "t_dt_note_off", "note_lng", "IOI", "prec_rest", "fllw_rest"]
    total_delta_time = 0
    notes_list = []
    note_on_counter = -1
    
    notes_list_pd_df = pd.DataFrame(columns=columns_names)
    
    target_track_index = -1 
    for idx, track in enumerate(midifile.tracks):
        for msg in track:
            if msg.type == 'note_on' and msg.velocity > 0:
                target_track_index = idx 
                break
        if target_track_index != -1:
            break 

    if target_track_index == -1:
        print(f"エラー: MIDIファイル内のどのトラックにもノート（音符）が見つかりませんでした。")
        print("処理を中断します。.dat ファイルは生成されません。")
        return 
    
    line = f"// Note: MIDIトラック {target_track_index + 1} (全 {num_midi_tracks} トラック中) を解析対象とします。"
    print(line); dat_content_lines.append(line)

    for counter in range(len(midifile.tracks[target_track_index])):
        
        try:  
            midifile.tracks[target_track_index][counter].time
        except:
            pass
        else:
            total_delta_time += midifile.tracks[target_track_index][counter].time

        try:  
            midifile.tracks[target_track_index][counter].note
        except:
            pass
        else:  
            if (midifile.tracks[target_track_index][counter].type == "note_on" and midifile.tracks[target_track_index][counter].velocity != 0):         
                note_on_counter += 1
                notes_list.append([midifile.tracks[target_track_index][counter].note, notes_names[midifile.tracks[target_track_index][counter].note], midifile.tracks[target_track_index][counter].time, total_delta_time, "tbp", "tbp", "tbp", "tbp", "tbp", "tbp"])
                if len(notes_list) == 1:
                        notes_list[0][7] = total_delta_time
                if len(notes_list) > 1:
                        notes_list[len(notes_list) - 1][7] = (total_delta_time - notes_list[len(notes_list) - 2][3])                     
                notes_list[note_on_counter][8] = notes_list[note_on_counter][2]        
                if len(notes_list) > 1:
                    notes_list[note_on_counter - 1][9] = notes_list[note_on_counter][8]
            elif midifile.tracks[target_track_index][counter].type == "note_off" or midifile.tracks[target_track_index][counter].velocity == 0:
                for v in range(len(notes_list)):
                    if (notes_list[v][0] == midifile.tracks[target_track_index][counter].note) and (
                            notes_list[v][4] == "tbp"):
                        notes_list[v][4] = midifile.tracks[target_track_index][counter].time
                        notes_list[v][5] = total_delta_time
                        notes_list[v][6] = notes_list[v][5] - notes_list[v][3]
                        break
            
            notes_list_pd_df = pd.DataFrame(notes_list, columns=columns_names)

    if notes_list_pd_df.empty:
        print(f"エラー: MIDIトラック {target_track_index + 1} にノート（音符）が見つかりませんでした。")
        print("処理を中断します。.dat ファイルは生成されません。")
        return 
            
    notes_list_pd_df.iat[-1, 9] = 0   
    
    ms_per_tick = tempo/ticks_per_beat/1000
    
    notes_list_pd_df['t_dt_note_on_ms'] = np.rint(notes_list_pd_df['t_dt_note_on'] * ms_per_tick).astype(np.int64)
    notes_list_pd_df['t_dt_note_off_ms'] = np.rint(notes_list_pd_df['t_dt_note_off'] * ms_per_tick).astype(np.int64)
    
    # --- (★ 修正 ★) ---
    # "availableTime" を「前の音符の終わりから、次の音符の始まりまで」の時間 (休符) として計算します。
    
    # 1. 1行前の t_dt_note_off_ms (前の音符の終了時刻) を取得します。
    previous_note_off_ms = notes_list_pd_df['t_dt_note_off_ms'].shift(1)
    
    # 2. (現在の音符の開始時刻) - (前の音符の終了時刻) を計算します。
    #    これが「真の移動可能時間 (休符)」= "True Available Time" です。
    notes_list_pd_df['True_Available_Time_ms'] = notes_list_pd_df['t_dt_note_on_ms'] - previous_note_off_ms
    
    # 3. 負の値を0に丸めます (音符が重なっている(Legato)場合)
    notes_list_pd_df['True_Available_Time_ms'] = notes_list_pd_df['True_Available_Time_ms'].clip(lower=0)
    
    # 4. 最初の音符 (NaN) には 1000 (デフォルト値) を設定します。
    notes_list_pd_df['True_Available_Time_ms'] = notes_list_pd_df['True_Available_Time_ms'].fillna(1000).astype(np.int64)
    # --- (★ 修正完了 ★) ---

    
    # (★ 修正 ★) 以下のカラムは .dat 生成に必須ではないため、IOI_ms の計算は削除
    notes_list_pd_df['note_lng_ms'] = np.rint(notes_list_pd_df['note_lng'] * ms_per_tick).astype(np.int64)
    # notes_list_pd_df['IOI_ms'] = np.rint(notes_list_pd_df['IOI'] * ms_per_tick).astype(np.int64) # ← 削除
    notes_list_pd_df['prec_rest_ms'] = np.rint(notes_list_pd_df['prec_rest'] * ms_per_tick).astype(np.int64)
    notes_list_pd_df['bpm'] = ''
    notes_list_pd_df['ticks_per_beat'] = ''
    notes_list_pd_df['numerator'] = ''
    notes_list_pd_df['denominator'] = ''
    notes_list_pd_df.at[0, 'bpm'] = bpm
    notes_list_pd_df.at[0, 'ticks_per_beat'] = ticks_per_beat
    notes_list_pd_df.at[0, 'numerator'] = numerator
    notes_list_pd_df.at[0, 'denominator'] = denominator
    
    for i in range(4):
        cols = list(notes_list_pd_df.columns)
        cols = [cols[-1]] + cols[:-1]
        notes_list_pd_df = notes_list_pd_df[cols]
    
    number_of_notes = len(notes_list_pd_df.index) # m
    line = "// Number of notes: " + str(number_of_notes) + "\\n"
    print(line); dat_content_lines.append(line)
    
    try:
        with open('instrument_settings.txt', 'r') as inst:   
            instrument_settings = inst.read()
            print(instrument_settings)
            dat_content_lines.append(instrument_settings)
    except FileNotFoundError:
        line = "警告: 'instrument_settings.txt' が見つかりません。"
        print(line); dat_content_lines.append(f"// {line}")
    
    notes_list = notes_list_pd_df['note_name'].values.tolist()
    notes_list_string = ",".join(str(element) for element in notes_list)
    

    # --- (★ 修正 ★) ---
    # 'availableTime' のリストを、新しく計算した 'True_Available_Time_ms' (休符) から作成します。
    
    available_time_list = notes_list_pd_df['True_Available_Time_ms'].values.tolist()
    
    # (最初の音符の 1000ms は True_Available_Time_ms の計算中に fillna(1000) で設定済み)
    
    available_time_list_string = ",".join(str(element) for element in available_time_list)
    # --- (★ 修正完了 ★) ---

    
    melody_string = f"""
//---------------------------melody

m = {number_of_notes};

note = [{notes_list_string}];

availableTime = [{available_time_list_string}];
"""
    print(melody_string)
    dat_content_lines.append(melody_string)

    optimization_settings = ""
    try:
        with open('optimization_settings.txt', 'r') as opt:   
            optimization_settings = opt.read()
    except FileNotFoundError:
        line = "警告: 'optimization_settings.txt' が見つかりません。"
        print(line); dat_content_lines.append(f"// {line}")
        optimization_settings = "// 'optimization_settings.txt' が見つからなかったため、デフォルト設定を書き込めません。"

    customize_request = input("//Do you want to customize the configuration? Answer Y or N ")
    
    if customize_request.upper() == "Y":
        print("\\n--- 物理パラメータ(mm)の入力 ---")
        scaleLength = input("// 1. スケール長 (mm) (例: 635.0) >> ")
        stringSpacing = input("// 2. 弦の間隔 (mm) (例: 10.8) >> ")
        SP_max_mm = input("// 3. 指の最大広がり (mm) [中,薬,小指] (例: [40.0, 50.0, 60.0]) >> ")
        timePerMm = input("// 4. 1mmあたりの移動時間 (ms) (例: 0.5) >> ")
        fingerLengths_mm = input("// 5. 各指の長さ (mm) [人,中,薬,小] (例: [80.0, 85.0, 83.0, 75.0]) >> ")
        pc_threshold_mm = input("// 6. PCコストが固定になる閾値 (mm) (例: 50.0) >> ")
        
        print("\\n--- ペナルティ係数 (d_X) の入力 ---")
        d_PC = input("// 7. d_PC (ポジション変更) (例: 1.0) >> ")
        d_SC = input("// 8. d_SC (弦変更) (例: 1.0) >> ")
        d_HS = input("// 9. d_HS (手の広がり) (例: 2.0) >> ")
        d_DH = input("// 10. d_DH (到達コスト) (例: 0.5) >> ")
        d_NO = input("// 11. d_NO (開放弦) (例: 100.0) >> ")

    else:
        print(optimization_settings) 
        dat_content_lines.append(optimization_settings)
    
    
    final_dat_content = "\n".join(dat_content_lines)
    try:
        with open(dat_output_path, 'w', encoding='utf-8') as f:
            f.write(final_dat_content)
        print(f"\\n--- 成功 ---")
        print(f".dat ファイルが {dat_output_path} に保存されました。")
    except Exception as e:
        print(f"\\n--- エラー ---")
        print(f".dat ファイルの保存に失敗しました: {e}")
    
# --- 関数を実行 ---
midifile_df_create()

// File name: complex_solo
// Number of track in the MIDI file: 2
// Ticks per beat of the MIDI file: 480
// Tempo of the MIDI file: 315789
// Time signature: 4/4\n
--- BPM確認 ---


検出されたBPM: 190 (このままEnter, または新しいBPMを入力) >>  


// BPM: 190
// Note: MIDIトラック 1 (全 2 トラック中) を解析対象とします。
// Number of notes: 104\n
//--------------------------instrument
f_max = 21;
s_max = 6;
PlayableSounds  = {E2,F2,Fs2,G2,Gs2,A2,As2,B2,C3,Cs3,D3,Ds3,E3,F3,Fs3,G3,Gs3,A3,As3,B3,C4,Cs4,D4,Ds4,E4,F4,Fs4,G4,Gs4,A4,As4,B4,C5,Cs5,D5,Ds5,E5,F5,Fs5,G5,Gs5,A5,As5,B5,C6,Cs6,D6};
OpenStrings = [{6},{},{},{},{},{5},{},{},{},{},{4},{},{},{},{},{3},{},{},{},{2},{},{},{},{},{1},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{}];
Combinations = [{},{<6,1>},{<6,2>},{<6,3>},{<6,4>},{<6,5>},{<5,1>,<6,6>},{<5,2>,<6,7>},{<5,3>,<6,8>},{<5,4>,<6,9>},{<5,5>,<6,10>},{<4,1>,<5,6>,<6,11>},{<4,2>,<5,7>,<6,12>},{<4,3>,<5,8>,<6,13>},{<4,4>,<5,9>,<6,14>},{<4,5>,<5,10>,<6,15>},{<3,1>,<4,6>,<5,11>,<6,16>},{<3,2>,<4,7>,<5,12>,<6,17>},{<3,3>,<4,8>,<5,13>,<6,18>},{<3,4>,<4,9>,<5,14>,<6,19>},{<2,1>,<3,5>,<4,10>,<5,15>,<6,20>},{<2,2>,<3,6>,<4,11>,<5,16>,<6,21>},{<2,3>,<3,7>,<4,12>,<5,17>},{<2,4>,<3,8>,<4,13>,<5,18>},{<2,5>,<3,9>,<4,14>,<5,19>},{<1,1>,<2,

//Do you want to customize the configuration? Answer Y or N  N


//-------------------------configuration
connectedTimeThreshold = 10.0; 
M_value = 10000.0;
timeThresholdRatio = 1.5;
scaleLength = 635.0;
stringSpacing = 10.8;
SP_max_mm = [60.0,80.0,100.0];
natural_min_spread_mm = [20.0, 40.0, 60.0];
timePerMm = 0.5;

fingerLengths_mm = [70.0, 75.0, 65.0, 50.0];
// Finger difficulty multiplier (1.0=easiest, 1.5=mid, 2.0=ring, 2.5=pinky)
fingerDifficulty = [1.0, 1.5, 2.0, 2.5];
timeThresholdRatio = 1.5;
maxHighPositionMultiplier = 2.0;

d_PC = 1.0; 
d_SC = 1.5;  
d_HS = 1.0;
d_DH = 40.0;  
d_NO = 50.0; 
              
              
// --- (EXISTING) alpha（重み） ---
alpha_PC = 0.2; //position change
alpha_SC = 0.2; //string change
alpha_HS = 0.2; //hand spread
alpha_DH = 0.2; //distance from the handy position
alpha_NO = 0.2; //number of notes played on open strings
\n--- 成功 ---
.dat ファイルが datas/complex_solo (1).dat に保存されました。
